In [1]:
# !pip install --quiet pyspark==3.5.0 delta-spark==3.1.0

In [57]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip

# Build SparkSession with Delta Lake support
builder = (
    SparkSession.builder
    .appName("CryptoExperiments")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 3.5.0


In [58]:
CSV_PATH = "../data/btc_data.csv"

# I. Read CSV with header; let Spark infer types
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CSV_PATH)
)

print(f"Rows: {df.count()}  |  Columns: {len(df.columns)}")

Rows: 655886  |  Columns: 12


In [59]:
df.show(5)

+-------------------+-------+-------+-------+-------+---------+--------------------+---------------+------+--------------+---------------+------+
|          open_time|   open|   high|    low|  close|   volume|          close_time|   quote_volume|trades|taker_buy_base|taker_buy_quote|ignore|
+-------------------+-------+-------+-------+-------+---------+--------------------+---------------+------+--------------+---------------+------+
|2020-01-01 00:00:00|7195.24|7196.25|7178.64|7179.78|95.509133|2020-01-01 00:04:...|686317.13625177|  1127|     32.773245|235537.29504531|     0|
|2020-01-01 00:05:00|7179.76|7191.77| 7178.2|7191.07|59.365225|2020-01-01 00:09:...|426481.26036406|   631|     24.766513|  177935.618201|     0|
|2020-01-01 00:10:00|7193.15|7193.53|7180.24|7180.97| 48.06851|2020-01-01 00:14:...|345446.50301879|   694|     19.422283|139596.62168263|     0|
|2020-01-01 00:15:00|7180.97| 7186.4|7177.35|7178.29|32.192929|2020-01-01 00:19:...|231162.55542356|   576|     12.963258| 9

In [60]:
# filter the data from 2025 onwards
df_filtered = df.filter(F.col("open_time") >= "2025-01-01")

In [61]:
df_filtered.show(5)

+-------------------+--------+--------+--------+--------+--------+--------------------+---------------+------+--------------+---------------+------+
|          open_time|    open|    high|     low|   close|  volume|          close_time|   quote_volume|trades|taker_buy_base|taker_buy_quote|ignore|
+-------------------+--------+--------+--------+--------+--------+--------------------+---------------+------+--------------+---------------+------+
|2025-01-01 00:00:00| 93576.0|93702.15| 93537.5| 93661.2| 45.9416|2025-01-01 00:04:...|4301907.4838205|  7448|      16.87967|1580504.0603142|     0|
|2025-01-01 00:05:00| 93661.2|93678.02| 93600.0| 93607.1|71.94585|2025-01-01 00:09:...|6736120.6541571|  4165|      16.66761|1560680.2912027|     0|
|2025-01-01 00:10:00| 93607.1|93656.19|93489.03|93656.18|57.96928|2025-01-01 00:14:...|5423765.8623784|  8175|      29.65936|2774717.2751442|     0|
|2025-01-01 00:15:00|93656.19|93840.05|93614.95|93796.35|39.74456|2025-01-01 00:19:...|3727084.2190624|  8

In [62]:
df_filtered.schema.fieldNames()

['open_time',
 'open',
 'high',
 'low',
 'close',
 'volume',
 'close_time',
 'quote_volume',
 'trades',
 'taker_buy_base',
 'taker_buy_quote',
 'ignore']

In [63]:
# drop ignore column
df_filtered = df_filtered.drop("ignore")

In [66]:
df_filtered = df_filtered.withColumn("symbol", F.lit("BTCUSDT"))

In [67]:
df_filtered.show(5)

+-------------------+--------+--------+--------+--------+--------+--------------------+---------------+------+--------------+---------------+-------+
|          open_time|    open|    high|     low|   close|  volume|          close_time|   quote_volume|trades|taker_buy_base|taker_buy_quote| symbol|
+-------------------+--------+--------+--------+--------+--------+--------------------+---------------+------+--------------+---------------+-------+
|2025-01-01 00:00:00| 93576.0|93702.15| 93537.5| 93661.2| 45.9416|2025-01-01 00:04:...|4301907.4838205|  7448|      16.87967|1580504.0603142|BTCUSDT|
|2025-01-01 00:05:00| 93661.2|93678.02| 93600.0| 93607.1|71.94585|2025-01-01 00:09:...|6736120.6541571|  4165|      16.66761|1560680.2912027|BTCUSDT|
|2025-01-01 00:10:00| 93607.1|93656.19|93489.03|93656.18|57.96928|2025-01-01 00:14:...|5423765.8623784|  8175|      29.65936|2774717.2751442|BTCUSDT|
|2025-01-01 00:15:00|93656.19|93840.05|93614.95|93796.35|39.74456|2025-01-01 00:19:...|3727084.21906

In [68]:
# II. Write to Delta Lake format in medallion architecture in bronze
DELTA_BRONZE_PATH = "../medallion/bronze/market"
df_filtered.write.format("delta") \
  .partitionBy("symbol") \
  .mode("append") \
  .save("../medallion/bronze/market/")

# Silver Table

In [139]:
DELTA_BRONZE_PATH = "../medallion/bronze/market"
df = spark.read.format("delta").load(DELTA_BRONZE_PATH)

In [140]:
df = df.repartition("symbol")

In [141]:
df.show(5)

+-------------------+---------+---------+---------+---------+--------+--------------------+---------------+------+--------------+---------------+-------+
|          open_time|     open|     high|      low|    close|  volume|          close_time|   quote_volume|trades|taker_buy_base|taker_buy_quote| symbol|
+-------------------+---------+---------+---------+---------+--------+--------------------+---------------+------+--------------+---------------+-------+
|2025-06-10 20:25:00|109573.85|109581.68| 109530.0|109537.41|18.33389|2025-06-10 20:29:...| 2008609.177547|  5558|       8.30732| 910087.9782644|BTCUSDT|
|2025-06-10 20:30:00|109537.41|109562.92|109462.54|109562.91| 20.2954|2025-06-10 20:34:...|2222475.0947494|  6780|        7.0192| 768681.4398297|BTCUSDT|
|2025-06-10 20:35:00|109562.92| 109728.1|109551.34|109717.13|35.70022|2025-06-10 20:39:...|3914073.4141862|  8770|      25.70654|2818527.9197048|BTCUSDT|
|2025-06-10 20:40:00|109717.13|109732.14|109636.93|109699.88|36.93931|2025-0

In [142]:
# Create new feature 
# log_return = log((close_price - open_price) / open_price)
df = df.withColumn("log_return", F.log(F.col("close") / F.col("open")))
# volatility = high - low 
df = df.withColumn("volatility", F.col("high") - F.col("low"))
# imbalance = taker_buy_base - (volume - taker_buy_base)
df = df.withColumn(
    "imbalance_ratio",
    (F.col("taker_buy_base") - (F.col("volume") - F.col("taker_buy_base"))) / F.col("volume")
)
# buy_ratio = taker_buy_base / volume
df = df.withColumn("buy_ratio", F.col("taker_buy_base") / F.col("volume"))
# vwap = quote_volume / volume
df = df.withColumn("vwap", F.col("quote_volume") / F.col("volume"))
df.show(5)

+-------------------+---------+---------+---------+---------+--------+--------------------+---------------+------+--------------+---------------+-------+--------------------+------------------+--------------------+------------------+------------------+
|          open_time|     open|     high|      low|    close|  volume|          close_time|   quote_volume|trades|taker_buy_base|taker_buy_quote| symbol|          log_return|        volatility|     imbalance_ratio|         buy_ratio|              vwap|
+-------------------+---------+---------+---------+---------+--------+--------------------+---------------+------+--------------+---------------+-------+--------------------+------------------+--------------------+------------------+------------------+
|2025-06-10 20:25:00|109573.85|109581.68| 109530.0|109537.41|18.33389|2025-06-10 20:29:...| 2008609.177547|  5558|       8.30732| 910087.9782644|BTCUSDT|-3.32616409909360...|51.679999999993015|-0.09377442539471977|0.4531127873026401|109557.1

In [143]:
from pyspark.sql.window import Window

window = Window.partitionBy("symbol").orderBy("open_time")

df = df.withColumn("log_return_lag1", F.lag("log_return", 1).over(window))
df = df.withColumn("log_return_lag2", F.lag("log_return", 2).over(window))
df = df.withColumn("buy_ratio_lag1", F.lag("buy_ratio", 1).over(window))

In [144]:
window_5 = Window.partitionBy("symbol").orderBy("open_time").rowsBetween(-5, 0)
window_20 = Window.partitionBy("symbol").orderBy("open_time").rowsBetween(-20, 0)

df = df.withColumn("ma_5", F.avg("close").over(window_5))
df = df.withColumn("ma_20", F.avg("close").over(window_20))

df = df.withColumn("volatility_5", F.avg("volatility").over(window_5))
df = df.withColumn("volume_5", F.avg("volume").over(window_5))
df = df.withColumn("buy_ratio_5", F.avg("buy_ratio").over(window_5))

In [145]:
df = df.withColumn("momentum", F.col("close") - F.col("ma_5"))

In [146]:
df = df.withColumn("volume_spike", F.col("volume") / F.col("volume_5"))

In [147]:
df = df.withColumn("price_range_ratio", (F.col("high") - F.col("low")) / F.col("close"))
df = df.withColumn("body_size", (F.col("close") - F.col("open")) / F.col("close"))

In [148]:
df = df.withColumn("hour", F.hour(F.col("open_time")))
df = df.withColumn("day_of_week", F.dayofweek(F.col("open_time")))

In [149]:
df = df.withColumn("trend_strength", F.col("ma_5") - F.col("ma_20"))

In [150]:
df = df.withColumn("volatility_ratio", F.col("volatility") / F.col("volatility_5"))

In [151]:
# drop quote_volume, taker_buy_quote, open, high, low, close_time
df = df.drop("quote_volume", "taker_buy_quote", "open", "high", "low", "close_time")

In [152]:
df = df.withColumn(
    "target",
    F.lead("log_return", 1).over(
        Window.partitionBy("symbol").orderBy("open_time")
    )
)

In [153]:
df.show(5)

+-------------------+--------+--------+------+--------------+-------+--------------------+------------------+--------------------+-------------------+-----------------+--------------------+--------------------+-------------------+-----------------+-----------------+------------------+----------+-------------------+------------------+------------------+--------------------+--------------------+----+-----------+--------------+------------------+--------------------+
|          open_time|   close|  volume|trades|taker_buy_base| symbol|          log_return|        volatility|     imbalance_ratio|          buy_ratio|             vwap|     log_return_lag1|     log_return_lag2|     buy_ratio_lag1|             ma_5|            ma_20|      volatility_5|  volume_5|        buy_ratio_5|          momentum|      volume_spike|   price_range_ratio|           body_size|hour|day_of_week|trend_strength|  volatility_ratio|              target|
+-------------------+--------+--------+------+--------------+-

In [154]:
from pyspark.sql import functions as F

null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])

null_counts.show()

+---------+-----+------+------+--------------+------+----------+----------+---------------+---------+----+---------------+---------------+--------------+----+-----+------------+--------+-----------+--------+------------+-----------------+---------+----+-----------+--------------+----------------+------+
|open_time|close|volume|trades|taker_buy_base|symbol|log_return|volatility|imbalance_ratio|buy_ratio|vwap|log_return_lag1|log_return_lag2|buy_ratio_lag1|ma_5|ma_20|volatility_5|volume_5|buy_ratio_5|momentum|volume_spike|price_range_ratio|body_size|hour|day_of_week|trend_strength|volatility_ratio|target|
+---------+-----+------+------+--------------+------+----------+----------+---------------+---------+----+---------------+---------------+--------------+----+-----+------------+--------+-----------+--------+------------+-----------------+---------+----+-----------+--------------+----------------+------+
|        0|    0|     0|     0|             0|     0|         0|         0|          

In [155]:
df = df.dropna(subset=[
    "log_return_lag1",
    "log_return_lag2",
    "buy_ratio_lag1",
    "ma_5",
    "ma_20"
])

In [159]:
df = df.withColumnRenamed("open_time", "timestamp")

In [160]:
df.show(5)

+-------------------+--------+--------+------+--------------+-------+--------------------+------------------+--------------------+------------------+-----------------+--------------------+--------------------+-------------------+-----------------+-----------------+------------------+-----------------+------------------+-----------------+------------------+--------------------+--------------------+----+-----------+------------------+------------------+--------------------+
|          timestamp|   close|  volume|trades|taker_buy_base| symbol|          log_return|        volatility|     imbalance_ratio|         buy_ratio|             vwap|     log_return_lag1|     log_return_lag2|     buy_ratio_lag1|             ma_5|            ma_20|      volatility_5|         volume_5|       buy_ratio_5|         momentum|      volume_spike|   price_range_ratio|           body_size|hour|day_of_week|    trend_strength|  volatility_ratio|              target|
+-------------------+--------+--------+------+

In [162]:
df.schema.fieldNames()

['timestamp',
 'close',
 'volume',
 'trades',
 'taker_buy_base',
 'symbol',
 'log_return',
 'volatility',
 'imbalance_ratio',
 'buy_ratio',
 'vwap',
 'log_return_lag1',
 'log_return_lag2',
 'buy_ratio_lag1',
 'ma_5',
 'ma_20',
 'volatility_5',
 'volume_5',
 'buy_ratio_5',
 'momentum',
 'volume_spike',
 'price_range_ratio',
 'body_size',
 'hour',
 'day_of_week',
 'trend_strength',
 'volatility_ratio',
 'target']

In [161]:
# Write to Delta Lake format in medallion architecture in silver
DELTA_SILVER_PATH = "../medallion/silver/market"
df.write.format("delta") \
  .partitionBy("symbol") \
  .mode("append") \
  .save(DELTA_SILVER_PATH)